In [1]:
%pip uninstall fitz pymupdf -y

Found existing installation: PyMuPDF 1.27.1
Uninstalling PyMuPDF-1.27.1:
  Successfully uninstalled PyMuPDF-1.27.1
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install pymupdf

  Using cached pymupdf-1.27.1-cp310-abi3-win_amd64.whl.metadata (3.4 kB)
Using cached pymupdf-1.27.1-cp310-abi3-win_amd64.whl (19.2 MB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
import fitz

In [4]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install dotenv

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install google-genai

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import re
import json
import fitz  # PyMuPDF
import ollama
from typing import List, Dict, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from google import genai

In [8]:
%pip install transformers 

Note: you may need to restart the kernel to use updated packages.


In [23]:
%pip install pydantic

Note: you may need to restart the kernel to use updated packages.


In [27]:
%pip install typing

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26419 sha256=47f53e874c6559d636788047614926352ff2a6cff601f7c51e681c0b8e91ff02
  Stored in directory: c:\users\user\appdata\local\pip\cache\wheels\12\98\52\2bffe242a9a487f00886e43b8ed8dac46456702e11a0d6abef
Successfully built typing
Note: you may need to restart the kernel to use updated packages.


In [28]:
from pydantic import field_validator

In [29]:
# Load configuration
load_dotenv()

True

In [30]:
from typing import Any, Union, List, Dict, Optional
from pydantic import BaseModel, Field, field_validator

In [39]:
# --- 1. DATA MODEL (Matches Assignment Requirements) ---
class DDRReport(BaseModel):
    property_issue_summary: Any
    probable_root_cause: Any
    severity_assessment: Any
    recommended_actions: Any
    additional_notes: Any
    conflicts: Any
    missing_information: Any
    confidence_score: Any

    @field_validator('*', mode='after')
    @classmethod
    def force_clean_string(cls, v):
        """Ensures the report is client-friendly and string-based, even if AI returns lists."""
        if isinstance(v, list):
            return " ".join([str(i) for i in v]) if v else "Not Available"
        if not v or str(v).strip() == "":
            return "Not Available"
        return str(v)

In [41]:
# --- 2. THE PRODUCTION PIPELINE ---
class DDRPipeline:
    def __init__(self, model_name="deepseek-r1:8b"):
        self.model = model_name

    def extract_text(self, pdf_path: str) -> str:
        """Robust text extraction."""
        try:
            with fitz.open(pdf_path) as doc:
                return "".join([page.get_text() for page in doc])
        except Exception as e:
            return f"Error: {e}"

    def parse_thermal_metrics(self, text: str) -> List[Dict]:
        """Extracts specific thermal data points for logical merging."""
        pattern = r"Thermal image\s*:\s*(.*?)\s.*?Hotspot\s*:\s*(\d+\.?\d*)\s*°C.*?Coldspot\s*:\s*(\d+\.?\d*)"
        matches = re.findall(pattern, text, re.DOTALL)
        return [{
            "id": m[0].strip(),
            "delta": round(float(m[1]) - float(m[2]), 2),
            "is_anomaly": (float(m[1]) - float(m[2])) > 4.0
        } for m in matches]

    def get_structured_report(self, inspection_text: str, thermal_data: list) -> DDRReport:
        """The 'Reasoning' step: Synthesizes two sources into one DDR."""
        
        # Prepare data for prompt
        anomalies = [d for d in thermal_data if d['is_anomaly']]
        
        prompt = f"""
        Analyze the following inspection documents and generate a final structured DDR.
        
        INSPECTION NOTES:
        {inspection_text[:6000]}
        
        THERMAL DATA ANOMALIES:
        {json.dumps(anomalies)}

        RULES FROM ASSIGNMENT GUIDE:
        1. If information conflicts (e.g., damp reported but thermal delta is low), mention it in 'conflicts'.
        2. If info is missing, write 'Not Available'.
        3. Use simple, client-friendly language. No technical jargon.
        4. Do NOT invent facts.

        OUTPUT JSON STRUCTURE:
        - "property_issue_summary": Overview of what's wrong.
        - "probable_root_cause": Why is this happening?
        - "severity_assessment": High/Medium/Low.
        - "recommended_actions": What should the client do next?
        - "additional_notes": Any other context.
        - "conflicts": List discrepancies between thermal and visual findings.
        - "missing_information": List what was not found in the documents.
        - "confidence_score": Percentage.
        """

        response = ollama.chat(
            model=self.model,
            messages=[{'role': 'user', 'content': prompt}],
            format="json",
            options={"temperature": 0} # Set to 0 for maximum reliability
        )
        
        # Clean output from DeepSeek tags
        content = response['message']['content']
        clean_json = re.sub(r'<tool_call>.*?<tool_call>', '', content, flags=re.DOTALL).strip()
        
        return DDRReport.model_validate_json(clean_json)

In [42]:
# --- 3. RUNNER ---
if __name__ == "__main__":
    # Update paths to your local files
    INSPECTION_FILE = "Sample_inputs/Sample Report.pdf"
    THERMAL_FILE = "Sample_inputs/Thermal Images.pdf"
    
    pipeline = DDRPipeline()

    print("Step 1: Reading Documents...")
    i_text = pipeline.extract_text(INSPECTION_FILE)
    t_text = pipeline.extract_text(THERMAL_FILE)
    
    print("Step 2: Analyzing Thermal Data...")
    t_metrics = pipeline.parse_thermal_metrics(t_text)

    print("Step 3: Reasoning & Generating DDR...")
    try:
        report = pipeline.get_structured_report(i_text, t_metrics)
        
        # Save output
        output_name = "final_ddr_report.json"
        with open(output_name, "w", encoding="utf-8") as f:
            f.write(report.model_dump_json(indent=4))
        
        print(f"\nSUCCESS: Report saved to {output_name}")
        print("-" * 30)
        print(f"Summary: {report.property_issue_summary[:100]}...")
        print(f"Conflicts: {report.conflicts}")
        print(f"Missing Info: {report.missing_information}")

    except Exception as e:
        print(f"Error: {e}")

Step 1: Reading Documents...
Step 2: Analyzing Thermal Data...
Step 3: Reasoning & Generating DDR...

SUCCESS: Report saved to final_ddr_report.json
------------------------------
Summary: The property has multiple dampness issues at skirting levels in Hall, Bedroom, Kitchen, and Master B...
Conflicts: Thermal anomalies detected despite visual confirmation of dampness, suggesting deeper issues not fully visible.
Missing Info: Exact cause of dampness Full moisture test results Details of plumbing issues
